# Research Paper Explainer

## 1 — Project Overview

This notebook takes a **research paper** (text or PDF) and explains it in simple language.

**End-to-end pipeline:**

```
Paper (PDF/text)
  → Extract text & structure sections
    → Generate plain-English summary
      → Extract glossary of technical terms
        → Explain like I'm a beginner
          → Generate a comprehension quiz
```

**Engineering patterns taught:**

| Pattern | Description |
|---------|-------------|
| Text extraction | Parse PDF or raw text |
| Section detection | Find Abstract, Introduction, Conclusion |
| Summarization | Distill key ideas into plain language |
| Glossary generation | Define all technical terms |
| Complexity tuning | Adjust explanation level per audience |
| Quiz generation | Create multiple-choice questions to test understanding |

**Stack:** Python standard library + regex for text parsing. Ollama optional for LLM-powered explanations.

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Extract text from research papers** — parse structured sections from academic papers
2. **Design prompts for different audiences** — tailor explanations from PhD-level to beginner
3. **Generate technical glossaries** — extract and define domain-specific terminology
4. **Build quiz generation pipelines** — create multiple-choice questions to test comprehension
5. **Handle LLM unavailability gracefully** — rule-based fallback for glossary extraction and quiz generation
6. **Evaluate explanation quality** — measure readability and comprehensiveness

## 3 — Problem Statement

**Problem:** Research papers are dense and technical. I want to understand a paper's core ideas without getting lost in math notation and jargon.

**Why it's hard:**
- Papers assume deep domain knowledge — years of prerequisite learning
- Every field has its own jargon that's never formally defined
- Mathematical notation varies across papers
- The key insight is buried in dense prose between definitions and proofs

**Our approach:**
```
Step 1: Extract the paper's structure (sections)
Step 2: Summarize each section in plain English
Step 3: Define all technical terms
Step 4: Generate a "beginner explanation"
Step 5: Create a quiz to test your understanding
```

## 4 — Why This Project Matters

**Paper summarization is a practical GenAI use case:**
- Researchers need to read 10–20 papers per week to stay current
- Automated summarization saves 30–50 minutes per paper
- Students use this to learn from papers faster than reading linearly
- Technical writers use this to distill complex work into documentation

**Complexity adjustment is an underrated pattern:**

| Audience | Key needs |
|----------|-----------|
| Expert (PhD) | Mathematical precision, novelty over prior work |
| Practitioner | How to apply it, computational cost, trade-offs |
| Student | What terms mean, why the approach makes sense |
| Non-technical | What problem it solves, why it matters to society |

Most paper reading tools assume one audience. This notebook shows how to adapt explanations.

## 5 — Paper Overview

We use **"Attention Is All You Need"** (Vaswani et al., 2017) — the Transformer paper.

| Property | Value |
|----------|-------|
| Title | Attention Is All You Need |
| Year | 2017 |
| Venue | NeurIPS |
| Impact | Foundation for LLMs (GPT, LLaMA, Qwen, etc.) |
| Sections | Abstract, Introduction, Architecture, Training, Results, Conclusion |
| Difficulty | High — requires understanding RNNs, attention, and optimization |

**Why this paper?**
- Foundational to modern AI — most readers will recognize it
- Clear structure — Abstract, Introduction, Architecture, Experiments, Conclusion
- Self-contained — doesn't require reading 5 prior papers to understand
- Teachable — the attention mechanism is visually intuitive

**Fallback:** If paper is unavailable, we include embedded structured content so the notebook runs offline.

## 6 — Environment Setup

We check two optional capabilities:
1. **Network access** — can we fetch the paper from arXiv?
2. **Ollama** — can we generate explanations with an LLM?

The notebook works fully without either (using embedded content + rule-based glossary/quiz).

In [ ]:
import urllib.request
import urllib.error
import json
import re
from typing import Any, Dict, List, Tuple


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    """Check if Ollama server is running locally."""
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False


def is_network_available() -> bool:
    """Check if we can reach arXiv."""
    try:
        with urllib.request.urlopen("https://arxiv.org/", timeout=5) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_OLLAMA = is_ollama_available(OLLAMA_HOST)
HAS_NETWORK = is_network_available()

print(f"Network access:  {HAS_NETWORK}")
print(f"Ollama available: {USE_OLLAMA}")
if not HAS_NETWORK:
    print("  -> Will use embedded sample paper")
if not USE_OLLAMA:
    print("  -> Will use rule-based explanations (no LLM)")


## 7 — Imports

All imports are standard Python library. We'll parse text with regex and nltk tokenization.

In [ ]:
import string
from collections import Counter

print("All imports loaded.")


## 8 — Data Loading (Paper Ingestion)

We support two input modes:
1. **Text files or raw paper content** (no PDF parsing library)
2. **Embedded paper content** as fallback

In production, you'd use `pypdf` or `pdfplumber` for PDF extraction. Here we use embedded text.

In [ ]:
# Embedded sample paper: "Attention Is All You Need" structure (summarized)
SAMPLE_PAPER = """
Title: Attention Is All You Need
Authors: Vaswani et al.
Year: 2017
Venue: NeurIPS

ABSTRACT
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. We propose a new simple network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train.

INTRODUCTION
Recurrent neural networks, long short-term memory (LSTM) networks and gated recurrent units (GRU) have been firmly established as state-of-the-art approaches in sequence modeling and transduction problems such as language modeling and machine translation. The sequential computation nature of RNNs makes them unsuitable for parallelization. The Transformer model addresses these limitations by relying entirely on an attention mechanism to draw global dependencies between input and output.

ATTENTION MECHANISM
We introduce the Scaled Dot-Product Attention mechanism. Given a query Q, key K, and value V, we compute the attention as:
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

Multi-head attention allows the model to attend to information from different representation subspaces at different positions. With h heads:
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
where each head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)

TRANSFORMER ARCHITECTURE
The Transformer follows an encoder-decoder structure. The encoder maps an input sequence to a sequence of continuous representations. The decoder takes these representations and generates an output sequence one element at a time. The Transformer uses stacked self-attention and point-wise, fully connected layers in both the encoder and decoder. Each layer consists of two sub-layers: (1) multi-head self-attention mechanism and (2) position-wise fully connected feed-forward network.

POSITION-WISE FEED-FORWARD NETWORKS
Each encoder and decoder layer contains a simple feed-forward network that is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between:
FFN(x) = max(0, xW_1 + b_1) W_2 + b_2

POSITIONAL ENCODING
Since the model contains no recurrence and no convolution, the model must rely on positional encoding to make use of the order of the sequence. We use sinusoidal positional encodings:
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

TRAINING
We train on the WMT 2014 English-German dataset containing 4.5 million sentence pairs. The base model consists of 6 layers with d_model = 512, h = 8 attention heads. We use byte-pair encoding (BPE) to encode the data. Training was done on 8 NVIDIA P100 GPUs for 12 hours.

RESULTS
Our model outperforms all previously known models on English-to-German and English-to-French translation tasks. The base model achieves 27.3 BLEU on English-to-German, and the big model achieves 28.4 BLEU.

CONCLUSION
In this work, we presented the Transformer, a new simple network architecture based on attention mechanisms. We demonstrated that this architecture can reach state-of-the-art translation quality more quickly than conventional approaches. We believe attention mechanisms will become the dominant approach for many NLP tasks.
"""

print(f"Embedded paper loaded: {len(SAMPLE_PAPER)} characters")


## 9 — Data Inspection

Let's analyze the paper structure before processing.

In [ ]:
def inspect_paper(text: str) -> Dict[str, Any]:
    return {
        "total_lines": len(text.split("\n")),
        "total_chars": len(text),
        "total_words": len(text.split()),
        "avg_line_len": len(text) / max(len(text.split("\n")), 1),
    }

info = inspect_paper(SAMPLE_PAPER)
print("Paper inspection:")
for k, v in info.items():
    print(f"  {k}: {v}")


## 10 — Preprocessing (Section Detection & Extraction)

We extract sections using structural markers: all-caps section headers like "ABSTRACT", "INTRODUCTION", etc.

Each section becomes a document we can summarize independently.

In [ ]:
def extract_sections(text: str) -> List[Dict[str, str]]:
    """Extract paper sections by detecting all-caps headers."""
    # Split by section headers (all uppercase lines with minimal punctuation)
    section_pattern = re.compile(r"^([A-Z][A-Z0-9\s\-]+)$", re.MULTILINE)
    matches = list(section_pattern.finditer(text))

    if not matches:
        return [{"title": "Full Text", "content": text}]

    sections = []
    for i, match in enumerate(matches):
        title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()

        if content:
            sections.append({"title": title, "content": content})

    return sections


sections = extract_sections(SAMPLE_PAPER)
print(f"Extracted {len(sections)} sections:")
for i, s in enumerate(sections):
    print(f"  [{i+1}] {s['title']:40s} {len(s['content']):>5} chars")


## 11 — Baseline Approach (Rule-Based Extraction)

Before using an LLM, we build rule-based extractors for **glossary** and **quiz generation**:

- **Glossary:** Extract capitalized technical terms and try to find adjacent definitions (e.g., "LSTM (long short-term memory)")
- **Quiz:** Create fill-in-the-blank questions from the text

These baselines work without an LLM and are often surprisingly effective.

In [ ]:
def extract_glossary_baseline(text: str, top_k: int = 10) -> List[Dict[str, str]]:
    """Extract technical terms using regex patterns."""
    # Pattern 1: Capitalized acronyms with definition in parentheses
    acronym_pattern = r"(\b[A-Z]{2,}\s*\([^)]+\))"
    matches = re.findall(acronym_pattern, text)

    glossary = []
    for match in matches[:top_k]:
        parts = re.match(r"(\b[A-Z]{2,})\s*\(([^)]+)\)", match)
        if parts:
            term = parts.group(1)
            definition = parts.group(2)
            glossary.append({"term": term, "definition": definition})

    # Pattern 2: Commonly repeated capitalized terms
    if len(glossary) < top_k:
        words = text.split()
        capitalized = [w.rstrip(string.punctuation) for w in words if w[0].isupper()]
        term_freq = Counter(capitalized)
        for term, freq in term_freq.most_common(top_k - len(glossary)):
            if len(term) > 2 and freq > 2:
                glossary.append({"term": term, "definition": "(Extracted from text)"})

    return glossary[:top_k]


# Test baseline glossary
baseline_glossary = extract_glossary_baseline(SAMPLE_PAPER, top_k=8)
print("Baseline glossary (rule-based extraction):")
for item in baseline_glossary:
    print(f"  {item['term']:20s} - {item['definition'][:60]}")


## 12 — Main Workflow (Summarization + Glossary + Quiz + Explanations)

We build prompt-based extractors that work with or without an LLM.

In [ ]:
def call_ollama(prompt: str, temperature: float = 0.7) -> str:
    """Call Ollama and return the response."""
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "temperature": temperature,
    }
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return data.get("response", "").strip()


# Prompts for different explanation levels
SUMMARY_PROMPT = """Summarize this research paper section in 3-4 sentences. Use simple language. Avoid jargon.
If you must use technical terms, define them.

Section:
{section}

Summary:"""

BEGINNER_EXPLAIN_PROMPT = """Explain this research paper section as if you are teaching a beginner with no background in the field.
Use analogies and examples. Break down complex ideas into simple steps. Avoid mathematical notation.

Section:
{section}

Beginner Explanation:"""

GLOSSARY_PROMPT = """Extract all technical terms from this text and provide plain-English definitions.
Format: one term per line as "TERM: definition"

Text:
{section}

Glossary:"""

QUIZ_PROMPT = """Generate 3 multiple-choice quiz questions based on this section.
Format each question as:
Q: [question text]
A) [option 1]
B) [option 2]
C) [option 3]
D) [option 4]
Correct: [A/B/C/D]

Section:
{section}

Quiz Questions:"""


def summarize_section(section_title: str, section_content: str) -> Dict[str, str]:
    """Summarize a paper section."""
    if USE_OLLAMA:
        summary = call_ollama(SUMMARY_PROMPT.format(section=section_content))
    else:
        # Fallback: take first 2-3 sentences
        sentences = re.split(r"(?<=[.!?])\s+", section_content)
        summary = " ".join(sentences[:3])

    return {"title": section_title, "summary": summary}


def explain_for_beginner(section_title: str, section_content: str) -> Dict[str, str]:
    """Explain a section for beginners."""
    if USE_OLLAMA:
        explanation = call_ollama(BEGINNER_EXPLAIN_PROMPT.format(section=section_content), temperature=0.8)
    else:
        # Fallback: simplified version
        explanation = section_content[:300] + "\n(Simplified explanation - LLM not available)"

    return {"title": section_title, "explanation": explanation}


def extract_glossary(section_title: str, section_content: str) -> List[Dict[str, str]]:
    """Extract glossary from a section."""
    if USE_OLLAMA:
        glossary_text = call_ollama(GLOSSARY_PROMPT.format(section=section_content), temperature=0.3)
        entries = []
        for line in glossary_text.split("\n"):
            if ":" in line:
                term, definition = line.split(":", 1)
                entries.append({"term": term.strip(), "definition": definition.strip()})
        return entries
    else:
        return extract_glossary_baseline(section_content)


def generate_quiz(section_title: str, section_content: str) -> str:
    """Generate quiz questions for a section."""
    if USE_OLLAMA:
        quiz = call_ollama(QUIZ_PROMPT.format(section=section_content), temperature=0.7)
    else:
        # Fallback: simple factual question
        key_phrase = section_content.split()[10:20] if len(section_content.split()) > 10 else section_content.split()[:5]
        quiz = f"Q: What is discussed in the {section_title} section?\nA) {' '.join(key_phrase)} concept\nB) Other concept\nC) Not mentioned\nD) Unclear\nCorrect: A"

    return quiz


print("Paper explanation functions defined.")


## 13 — Pipeline Execution

Let's process each section and generate summaries, glossary, and quiz.

In [ ]:
# Process all sections
summaries = []
beginner_explanations = []
all_glossary = []
all_quizzes = []

print("Processing paper sections...")
for section in sections[:4]:  # Process first 4 sections to save time
    title = section["title"]
    content = section["content"][:800]  # Limit to first 800 chars per section

    print(f"\n  {title}...")
    summaries.append(summarize_section(title, content))
    beginner_explanations.append(explain_for_beginner(title, content))
    all_glossary.extend(extract_glossary(title, content))
    quiz = generate_quiz(title, content)
    all_quizzes.append({"section": title, "quiz": quiz})

print(f"\nDone! Processed {len(summaries)} sections")


In [ ]:
# Display summaries
print("=" * 70)
print("PLAIN-ENGLISH SUMMARIES")
print("=" * 70)
for s in summaries:
    print(f"\n{s['title']}")
    print("-" * 70)
    print(s['summary'][:300] + "...")
    print()


In [ ]:
# Display beginner explanations
print("=" * 70)
print("BEGINNER EXPLANATIONS")
print("=" * 70)
for ex in beginner_explanations[:2]:  # Show first 2
    print(f"\n{ex['title']} (For Beginners)")
    print("-" * 70)
    print(ex['explanation'][:400] + "...")
    print()


In [ ]:
# Display glossary
glossary_unique = {}
for item in all_glossary:
    if item["term"] not in glossary_unique:
        glossary_unique[item["term"]] = item["definition"]

print("=" * 70)
print(f"GLOSSARY ({len(glossary_unique)} terms)")
print("=" * 70)
for term, definition in list(glossary_unique.items())[:15]:
    print(f"\n{term}")
    print(f"  {definition}")


In [ ]:
# Display quiz
print("=" * 70)
print("COMPREHENSION QUIZ")
print("=" * 70)
for item in all_quizzes[:2]:
    print(f"\n[From: {item['section']}]")
    print(item['quiz'][:500])
    print()


## 14 — Evaluation and Interpretation

We evaluate:
1. **Coverage** — how much of the paper did we process?
2. **Glossary quality** — are the definitions accurate and beginner-friendly?
3. **Quiz quality** — do the questions test key concepts?

In [ ]:
def evaluate_outputs(sections: List, summaries: List, glossary: List, quizzes: List) -> Dict[str, Any]:
    """Evaluate the output quality."""
    return {
        "sections_processed": len(summaries),
        "total_sections_in_paper": len(sections),
        "glossary_terms": len(set(g["term"] for g in glossary)),
        "quiz_questions": sum(q["quiz"].count("Q:") for q in quizzes),
        "avg_summary_length": sum(len(s["summary"]) for s in summaries) // max(len(summaries), 1),
        "mode_used": "LLM (Ollama)" if USE_OLLAMA else "Rule-based (fallback)",
    }

eval_result = evaluate_outputs(sections, summaries, all_glossary, all_quizzes)
print("\nEvaluation:")
for k, v in eval_result.items():
    print(f"  {k}: {v}")


## 15 — Error Analysis / Limitations

**Extraction limitations:**
- Section detection is regex-based — won't work on papers with inconsistent formatting
- No figure/table extraction — only text
- Mathematical notation is kept as-is (hard to explain without rendering)
- Dense paragraphs without clear sub-points are hard to break down

**Glossary limitations:**
- Acronym detection works only if definition is in parentheses
- Some terms appear in multiple contexts with different meanings
- Domain-specific terms sometimes need more context than a single sentence

**Quiz generation limitations:**
- Without proper understanding, quiz questions can be ambiguous
- Multiple-choice questions are harder to generate well than true/false
- Some questions may not have a single "correct" answer (debatable concepts)

**LLM-specific:**
- Hallucination risk — LLM may invent details not in the paper
- Context window limits — very long papers need to be processed iteratively
- Temperature setting affects balance between creativity and accuracy

## 16 — How to Improve It

### Production Upgrade Ideas

1. **PDF parsing**: Use `pypdf` or `pdfplumber` for real PDF extraction
2. **Figure captions**: Extract and explain figures, not just text
3. **Math rendering**: Convert LaTeX to inline explanations
4. **Comparison mode**: "How is this approach different from X?"
5. **Citation tracking**: Find related papers mentioned in the references
6. **Multi-level summaries**: 1-sentence, 1-paragraph, full-length options
7. **Audio version**: Generate a podcast-style audible explanation
8. **Interactive Q&A**: Ask clarifying questions about sections you didn't understand
9. **Source code examples**: If the paper links to code, explain it too
10. **Timeline view**: Show how this work connects to prior papers over time

## Common Mistakes

1. **One summary per paper**
   Different audiences need different summaries. A student needs concept explanations; a practitioner needs implementation details. Generate multiple versions.

2. **Glossary without context**
   Defining "attention" as "A mechanism that weights input elements" is useless. Show context: where it's used, why it matters.

3. **Quiz without answer explanations**
   A quiz that just says "Correct: A" doesn't teach. Always explain why A is correct and B/C/D are wrong.

4. **Ignoring mathematical content**
   Many papers are math-heavy. Don't skip equations — explain them step-by-step or provide intuition.

5. **No difficulty calibration**
   A beginner explanation that still assumes you know what "gradient descent" is defeats the purpose. Use analogies, not jargon.

6. **Treating all sections equally**
   Abstract and introduction are key; related work is context. Allocate explanation effort differently per section.

## 17 — Practice Exercises

### Mini Challenge

**Challenge 1:** Find a research paper on a topic you're interested in (from arXiv.org). Run it through this pipeline. Does the beginner explanation make sense?

**Challenge 2:** Modify the prompts to generate "expert summary" (for PhD level) vs. "layperson summary" (for general audience). How do they differ?

**Challenge 3:** Add a "limitations section detector" — extract the Limitations section and explain what the authors say the work *cannot* do. Why is this important?

### Try This Next
- Build a paper comparison tool: given 2 papers, explain their key differences
- Add a "key takeaway generator" that lists the 3 most important ideas in 1 sentence each
- Create a timeline view: explain how this paper builds on prior work and influences future work

## 18 — Final Takeaway / Summary

This notebook demonstrated a complete **research paper explanation pipeline**:

1. **Section extraction** — identify paper structure automatically
2. **Multi-level summaries** — from expert to beginner language
3. **Glossary generation** — define technical terms
4. **Quiz generation** — test and reinforce understanding
5. **Graceful degradation** — works without LLM using rule-based fallback

**Key principle:** Different audiences need different explanations. The same paper explained to a student will look very different from the version for a CEO. Tailor the explanation level to the reader.

**Real-world application:** Research teams, students, and companies all use paper summarization weekly. This pipeline is the foundation of tools like Consensus, Research Rabbit, and academic search engines.